In [1]:
!pwd

/content


In [2]:
import sys;
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [3]:
# !pip install uv

# !uv pip install \
#   "darts==0.34.0" \
#   "torch==2.3.1" \
#   "pytorch-lightning==2.2.5" \
#   "torchmetrics==1.3.2" \
#   "numpy==1.26.4" \
#   "pandas==2.2.2" \
#   "scikit-learn==1.7.2" \
#   "dask==2025.5.0" \
#   "xgboost==3.0.2" \
#   "catboost==1.2.8" \
#   "pyyaml==6.0.2" \
#   "lightgbm==4.6.0"

# Pin torch for reproducibility with your local run
# !uv pip install -q --index-url https://download.pytorch.org/whl/cu121 "torch==2.3.1"

In [4]:
import sys, torch, darts, pytorch_lightning as pl, numpy as np, pandas as pd
print(sys.version)
print("torch:", torch.__version__)
print("darts:", darts.__version__)
print("lightning:", pl.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("cuda:", torch.cuda.is_available())

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch: 2.3.1+cu121
darts: 0.34.0
lightning: 2.2.5
numpy: 1.26.4
pandas: 2.2.2
cuda: False


**Forecasts**
 - **Country: Canada**
 - **Forecast Horizons:12M and 24M Forecast**

In [5]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import XGBModel, CatBoostModel
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error
import torch.nn as nn
import torch.nn.functional as F

class DishTS(nn.Module):
  """DishTS Module for time series normalization"""
  def __init__(self, args):
      super().__init__()
      init = args['init']
      activate = True
      n_series = args['n_series']
      lookback = args['seq_len']

      if init == 'standard':
          self.reduce_mlayer = nn.Parameter(torch.rand(n_series, lookback, 2) / lookback)
      elif init == 'avg':
          self.reduce_mlayer = nn.Parameter(torch.ones(n_series, lookback, 2) / lookback)
      elif init == 'uniform':
          self.reduce_mlayer = nn.Parameter(torch.ones(n_series, lookback, 2) / lookback +
                                          torch.rand(n_series, lookback, 2) / lookback)

      self.gamma = nn.Parameter(torch.ones(n_series))
      self.beta = nn.Parameter(torch.zeros(n_series))
      self.activate = activate

  def forward(self, batch_x, mode='forward', dec_inp=None):
      if mode == 'forward':
          self.preget(batch_x)
          batch_x = self.forward_process(batch_x)
          dec_inp = None if dec_inp is None else self.forward_process(dec_inp)
          return batch_x, dec_inp
      elif mode == 'inverse':
          batch_y = self.inverse_process(batch_x)
          return batch_y

  def preget(self, batch_x):
      x_transpose = batch_x.permute(2, 0, 1)
      theta = torch.bmm(x_transpose, self.reduce_mlayer).permute(1, 2, 0)
      if self.activate:
          theta = F.gelu(theta)
      self.phil, self.phih = theta[:, :1, :], theta[:, 1:, :]
      self.xil = torch.sum(torch.pow(batch_x - self.phil, 2), axis=1, keepdim=True) / (batch_x.shape[1] - 1)
      self.xih = torch.sum(torch.pow(batch_x - self.phih, 2), axis=1, keepdim=True) / (batch_x.shape[1] - 1)

  def forward_process(self, batch_input):
      temp = (batch_input - self.phil) / torch.sqrt(self.xil + 1e-8)
      rst = temp.mul(self.gamma) + self.beta
      return rst

  def inverse_process(self, batch_input):
      return ((batch_input - self.beta) / self.gamma) * torch.sqrt(self.xih + 1e-8) + self.phih

class CANADAForecastGenerator:
  """Main class for generating forecasts using DishTS + XGB"""
  def __init__(self, data_path: str, random_seed: int = 1):
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      self.data = pd.read_csv(data_path)

      self.target_vars = [
          "Unemploymentrate",
          "RealbroadEER",
          "ShorttermIR",
          "OilpriceGlobalWTI",
          "CPIinflationrate"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int):
      """Prepare training data with correct lengths for past covariates"""
      # Split data
      train_size = len(self.data) - forecast_horizon
      train = self.data.head(train_size).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries using full training data
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(self.data[var]))

      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def generate_forecasts(self, forecast_horizons: list):
      """Generate forecasts for specified horizons"""
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data with correct lengths
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Initialize DishTS
          dish_args = {
              'init': 'standard',
              'n_series': len(self.target_vars),
              'seq_len': len(train_target_ts)
          }
          dish_ts = DishTS(dish_args)

          # Normalize training data
          train_normalized, _ = dish_ts(
              torch.tensor(train_target_ts.values(), dtype=torch.float32).unsqueeze(0)
          )

          # Create and train CatBoostModel with appropriate parameters
          model = CatBoostModel(
              lags = 6, #horizon,  # Set lags equal to horizon
              lags_past_covariates = 6, #horizon,  # Set past covariates lags equal to horizon
              output_chunk_length=horizon  # Set output chunk length equal to horizon
          )

          # Fit model with prepared data
          model.fit(
              TimeSeries.from_values(train_normalized.squeeze().detach().numpy()),
              past_covariates=train_exog_ts,
              # show_warnings=False
          )

          # Generate predictions
          pred_normalized = model.predict(horizon)

          # Inverse transform predictions
          with torch.no_grad():
              pred_original = dish_ts.inverse_process(
                  torch.tensor(pred_normalized.values(), dtype=torch.float32).unsqueeze(0)
              )

          # Create forecast DataFrame
          forecast_df = pd.DataFrame(
              np.squeeze(pred_original.detach().numpy()),
              columns=[f'forecast_{var.lower()}' for var in self.target_vars]
          )

          # Save forecasts
          output_file = f'canada_forecasts_DishTS_CATBOOST_{horizon}m.csv'
          forecast_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

          forecasts[horizon] = forecast_df

      return forecasts

def main():
  """Main function to run the forecast generation"""
  # Initialize forecast generator
  generator = CANADAForecastGenerator(
      data_path='all_mulvar_data_canada_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"canada_forecasts_DishTS_CATBOOST_{horizon}m.csv")

if __name__ == "__main__":
  main()


Generating 12-month forecast...
Forecast saved to canada_forecasts_DishTS_CATBOOST_12m.csv

Generating 24-month forecast...
Forecast saved to canada_forecasts_DishTS_CATBOOST_24m.csv

Created files:
canada_forecasts_DishTS_CATBOOST_12m.csv
canada_forecasts_DishTS_CATBOOST_24m.csv


**Forecasts**
 - **Country: USA**
 - **Forecast Horizons:12M and 24M Forecast**

In [6]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import XGBModel, CatBoostModel
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error
import torch.nn as nn
import torch.nn.functional as F

class DishTS(nn.Module):
  """DishTS Module for time series normalization"""
  def __init__(self, args):
      super().__init__()
      init = args['init']
      activate = True
      n_series = args['n_series']
      lookback = args['seq_len']

      if init == 'standard':
          self.reduce_mlayer = nn.Parameter(torch.rand(n_series, lookback, 2) / lookback)
      elif init == 'avg':
          self.reduce_mlayer = nn.Parameter(torch.ones(n_series, lookback, 2) / lookback)
      elif init == 'uniform':
          self.reduce_mlayer = nn.Parameter(torch.ones(n_series, lookback, 2) / lookback +
                                          torch.rand(n_series, lookback, 2) / lookback)

      self.gamma = nn.Parameter(torch.ones(n_series))
      self.beta = nn.Parameter(torch.zeros(n_series))
      self.activate = activate

  def forward(self, batch_x, mode='forward', dec_inp=None):
      if mode == 'forward':
          self.preget(batch_x)
          batch_x = self.forward_process(batch_x)
          dec_inp = None if dec_inp is None else self.forward_process(dec_inp)
          return batch_x, dec_inp
      elif mode == 'inverse':
          batch_y = self.inverse_process(batch_x)
          return batch_y

  def preget(self, batch_x):
      x_transpose = batch_x.permute(2, 0, 1)
      theta = torch.bmm(x_transpose, self.reduce_mlayer).permute(1, 2, 0)
      if self.activate:
          theta = F.gelu(theta)
      self.phil, self.phih = theta[:, :1, :], theta[:, 1:, :]
      self.xil = torch.sum(torch.pow(batch_x - self.phil, 2), axis=1, keepdim=True) / (batch_x.shape[1] - 1)
      self.xih = torch.sum(torch.pow(batch_x - self.phih, 2), axis=1, keepdim=True) / (batch_x.shape[1] - 1)

  def forward_process(self, batch_input):
      temp = (batch_input - self.phil) / torch.sqrt(self.xil + 1e-8)
      rst = temp.mul(self.gamma) + self.beta
      return rst

  def inverse_process(self, batch_input):
      return ((batch_input - self.beta) / self.gamma) * torch.sqrt(self.xih + 1e-8) + self.phih

class USAForecastGenerator:
  """Main class for generating forecasts using DishTS + XGB"""
  def __init__(self, data_path: str, random_seed: int = 1):
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      self.data = pd.read_csv(data_path)

      self.target_vars = [
          "Unemploymentrate",
          "RealbroadEER",
          "ShorttermIR",
          "OilpriceGlobalWTI",
          "CPIinflationrate"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int):
      """Prepare training data with correct lengths for past covariates"""
      # Split data
      train_size = len(self.data) - forecast_horizon
      train = self.data.head(train_size).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries using full training data
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(self.data[var]))

      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def generate_forecasts(self, forecast_horizons: list):
      """Generate forecasts for specified horizons"""
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data with correct lengths
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Initialize DishTS
          dish_args = {
              'init': 'standard',
              'n_series': len(self.target_vars),
              'seq_len': len(train_target_ts)
          }
          dish_ts = DishTS(dish_args)

          # Normalize training data
          train_normalized, _ = dish_ts(
              torch.tensor(train_target_ts.values(), dtype=torch.float32).unsqueeze(0)
          )

          # Create and train CatBoostModel with appropriate parameters
          model = CatBoostModel(
              lags = 6, #horizon,  # Set lags equal to horizon
              lags_past_covariates = 6, #horizon,  # Set past covariates lags equal to horizon
              output_chunk_length=horizon  # Set output chunk length equal to horizon
          )

          # Fit model with prepared data
          model.fit(
              TimeSeries.from_values(train_normalized.squeeze().detach().numpy()),
              past_covariates=train_exog_ts,
              # show_warnings=False
          )

          # Generate predictions
          pred_normalized = model.predict(horizon)

          # Inverse transform predictions
          with torch.no_grad():
              pred_original = dish_ts.inverse_process(
                  torch.tensor(pred_normalized.values(), dtype=torch.float32).unsqueeze(0)
              )

          # Create forecast DataFrame
          forecast_df = pd.DataFrame(
              np.squeeze(pred_original.detach().numpy()),
              columns=[f'forecast_{var.lower()}' for var in self.target_vars]
          )

          # Save forecasts
          output_file = f'usa_forecasts_DishTS_CATBOOST_{horizon}m.csv'
          forecast_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

          forecasts[horizon] = forecast_df

      return forecasts

def main():
  """Main function to run the forecast generation"""
  # Initialize forecast generator
  generator = USAForecastGenerator(
      data_path='all_mulvar_data_usa_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"usa_forecasts_DishTS_CATBOOST_{horizon}m.csv")

if __name__ == "__main__":
  main()


Generating 12-month forecast...
Forecast saved to usa_forecasts_DishTS_CATBOOST_12m.csv

Generating 24-month forecast...
Forecast saved to usa_forecasts_DishTS_CATBOOST_24m.csv

Created files:
usa_forecasts_DishTS_CATBOOST_12m.csv
usa_forecasts_DishTS_CATBOOST_24m.csv


**Forecasts**
 - **Country: France**
 - **Forecast Horizons:12M and 24M Forecast**

In [7]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import XGBModel, CatBoostModel
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error
import torch.nn as nn
import torch.nn.functional as F

class DishTS(nn.Module):
  """DishTS Module for time series normalization"""
  def __init__(self, args):
      super().__init__()
      init = args['init']
      activate = True
      n_series = args['n_series']
      lookback = args['seq_len']

      if init == 'standard':
          self.reduce_mlayer = nn.Parameter(torch.rand(n_series, lookback, 2) / lookback)
      elif init == 'avg':
          self.reduce_mlayer = nn.Parameter(torch.ones(n_series, lookback, 2) / lookback)
      elif init == 'uniform':
          self.reduce_mlayer = nn.Parameter(torch.ones(n_series, lookback, 2) / lookback +
                                          torch.rand(n_series, lookback, 2) / lookback)

      self.gamma = nn.Parameter(torch.ones(n_series))
      self.beta = nn.Parameter(torch.zeros(n_series))
      self.activate = activate

  def forward(self, batch_x, mode='forward', dec_inp=None):
      if mode == 'forward':
          self.preget(batch_x)
          batch_x = self.forward_process(batch_x)
          dec_inp = None if dec_inp is None else self.forward_process(dec_inp)
          return batch_x, dec_inp
      elif mode == 'inverse':
          batch_y = self.inverse_process(batch_x)
          return batch_y

  def preget(self, batch_x):
      x_transpose = batch_x.permute(2, 0, 1)
      theta = torch.bmm(x_transpose, self.reduce_mlayer).permute(1, 2, 0)
      if self.activate:
          theta = F.gelu(theta)
      self.phil, self.phih = theta[:, :1, :], theta[:, 1:, :]
      self.xil = torch.sum(torch.pow(batch_x - self.phil, 2), axis=1, keepdim=True) / (batch_x.shape[1] - 1)
      self.xih = torch.sum(torch.pow(batch_x - self.phih, 2), axis=1, keepdim=True) / (batch_x.shape[1] - 1)

  def forward_process(self, batch_input):
      temp = (batch_input - self.phil) / torch.sqrt(self.xil + 1e-8)
      rst = temp.mul(self.gamma) + self.beta
      return rst

  def inverse_process(self, batch_input):
      return ((batch_input - self.beta) / self.gamma) * torch.sqrt(self.xih + 1e-8) + self.phih

class FRANCEForecastGenerator:
  """Main class for generating forecasts using DishTS + XGB"""
  def __init__(self, data_path: str, random_seed: int = 1):
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      self.data = pd.read_csv(data_path)

      self.target_vars = [
          "Unemploymentrate",
          "RealbroadEER",
          "ShorttermIR",
          "OilpriceGlobalWTI",
          "CPIinflationrate"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int):
      """Prepare training data with correct lengths for past covariates"""
      # Split data
      train_size = len(self.data) - forecast_horizon
      train = self.data.head(train_size).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries using full training data
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(self.data[var]))

      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def generate_forecasts(self, forecast_horizons: list):
      """Generate forecasts for specified horizons"""
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data with correct lengths
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Initialize DishTS
          dish_args = {
              'init': 'standard',
              'n_series': len(self.target_vars),
              'seq_len': len(train_target_ts)
          }
          dish_ts = DishTS(dish_args)

          # Normalize training data
          train_normalized, _ = dish_ts(
              torch.tensor(train_target_ts.values(), dtype=torch.float32).unsqueeze(0)
          )

          # Create and train CatBoostModel with appropriate parameters
          model = CatBoostModel(
              lags = 6, #horizon,  # Set lags equal to horizon
              lags_past_covariates = 6, #horizon,  # Set past covariates lags equal to horizon
              output_chunk_length=horizon  # Set output chunk length equal to horizon
          )

          # Fit model with prepared data
          model.fit(
              TimeSeries.from_values(train_normalized.squeeze().detach().numpy()),
              past_covariates=train_exog_ts,
              # show_warnings=False
          )

          # Generate predictions
          pred_normalized = model.predict(horizon)

          # Inverse transform predictions
          with torch.no_grad():
              pred_original = dish_ts.inverse_process(
                  torch.tensor(pred_normalized.values(), dtype=torch.float32).unsqueeze(0)
              )

          # Create forecast DataFrame
          forecast_df = pd.DataFrame(
              np.squeeze(pred_original.detach().numpy()),
              columns=[f'forecast_{var.lower()}' for var in self.target_vars]
          )

          # Save forecasts
          output_file = f'france_forecasts_DishTS_CATBOOST_{horizon}m.csv'
          forecast_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

          forecasts[horizon] = forecast_df

      return forecasts

def main():
  """Main function to run the forecast generation"""
  # Initialize forecast generator
  generator = FRANCEForecastGenerator(
      data_path='all_mulvar_data_france_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"france_forecasts_DishTS_CATBOOST_{horizon}m.csv")

if __name__ == "__main__":
  main()


Generating 12-month forecast...
Forecast saved to france_forecasts_DishTS_CATBOOST_12m.csv

Generating 24-month forecast...
Forecast saved to france_forecasts_DishTS_CATBOOST_24m.csv

Created files:
france_forecasts_DishTS_CATBOOST_12m.csv
france_forecasts_DishTS_CATBOOST_24m.csv


**Forecasts**
 - **Country: Germany**
 - **Forecast Horizons:12M and 24M Forecast**

In [8]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import XGBModel, CatBoostModel
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error
import torch.nn as nn
import torch.nn.functional as F

class DishTS(nn.Module):
  """DishTS Module for time series normalization"""
  def __init__(self, args):
      super().__init__()
      init = args['init']
      activate = True
      n_series = args['n_series']
      lookback = args['seq_len']

      if init == 'standard':
          self.reduce_mlayer = nn.Parameter(torch.rand(n_series, lookback, 2) / lookback)
      elif init == 'avg':
          self.reduce_mlayer = nn.Parameter(torch.ones(n_series, lookback, 2) / lookback)
      elif init == 'uniform':
          self.reduce_mlayer = nn.Parameter(torch.ones(n_series, lookback, 2) / lookback +
                                          torch.rand(n_series, lookback, 2) / lookback)

      self.gamma = nn.Parameter(torch.ones(n_series))
      self.beta = nn.Parameter(torch.zeros(n_series))
      self.activate = activate

  def forward(self, batch_x, mode='forward', dec_inp=None):
      if mode == 'forward':
          self.preget(batch_x)
          batch_x = self.forward_process(batch_x)
          dec_inp = None if dec_inp is None else self.forward_process(dec_inp)
          return batch_x, dec_inp
      elif mode == 'inverse':
          batch_y = self.inverse_process(batch_x)
          return batch_y

  def preget(self, batch_x):
      x_transpose = batch_x.permute(2, 0, 1)
      theta = torch.bmm(x_transpose, self.reduce_mlayer).permute(1, 2, 0)
      if self.activate:
          theta = F.gelu(theta)
      self.phil, self.phih = theta[:, :1, :], theta[:, 1:, :]
      self.xil = torch.sum(torch.pow(batch_x - self.phil, 2), axis=1, keepdim=True) / (batch_x.shape[1] - 1)
      self.xih = torch.sum(torch.pow(batch_x - self.phih, 2), axis=1, keepdim=True) / (batch_x.shape[1] - 1)

  def forward_process(self, batch_input):
      temp = (batch_input - self.phil) / torch.sqrt(self.xil + 1e-8)
      rst = temp.mul(self.gamma) + self.beta
      return rst

  def inverse_process(self, batch_input):
      return ((batch_input - self.beta) / self.gamma) * torch.sqrt(self.xih + 1e-8) + self.phih

class GERMANYForecastGenerator:
  """Main class for generating forecasts using DishTS + XGB"""
  def __init__(self, data_path: str, random_seed: int = 1):
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      self.data = pd.read_csv(data_path)

      self.target_vars = [
          "Unemploymentrate",
          "RealbroadEER",
          "ShorttermIR",
          "OilpriceGlobalWTI",
          "CPIinflationrate"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int):
      """Prepare training data with correct lengths for past covariates"""
      # Split data
      train_size = len(self.data) - forecast_horizon
      train = self.data.head(train_size).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries using full training data
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(self.data[var]))

      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def generate_forecasts(self, forecast_horizons: list):
      """Generate forecasts for specified horizons"""
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data with correct lengths
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Initialize DishTS
          dish_args = {
              'init': 'standard',
              'n_series': len(self.target_vars),
              'seq_len': len(train_target_ts)
          }
          dish_ts = DishTS(dish_args)

          # Normalize training data
          train_normalized, _ = dish_ts(
              torch.tensor(train_target_ts.values(), dtype=torch.float32).unsqueeze(0)
          )

          # Create and train CatBoostModel with appropriate parameters
          model = CatBoostModel(
              lags = 6, #horizon,  # Set lags equal to horizon
              lags_past_covariates = 6, #horizon,  # Set past covariates lags equal to horizon
              output_chunk_length=horizon  # Set output chunk length equal to horizon
          )

          # Fit model with prepared data
          model.fit(
              TimeSeries.from_values(train_normalized.squeeze().detach().numpy()),
              past_covariates=train_exog_ts,
              # show_warnings=False
          )

          # Generate predictions
          pred_normalized = model.predict(horizon)

          # Inverse transform predictions
          with torch.no_grad():
              pred_original = dish_ts.inverse_process(
                  torch.tensor(pred_normalized.values(), dtype=torch.float32).unsqueeze(0)
              )

          # Create forecast DataFrame
          forecast_df = pd.DataFrame(
              np.squeeze(pred_original.detach().numpy()),
              columns=[f'forecast_{var.lower()}' for var in self.target_vars]
          )

          # Save forecasts
          output_file = f'germany_forecasts_DishTS_CATBOOST_{horizon}m.csv'
          forecast_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

          forecasts[horizon] = forecast_df

      return forecasts

def main():
  """Main function to run the forecast generation"""
  # Initialize forecast generator
  generator = GERMANYForecastGenerator(
      data_path='all_mulvar_data_germany_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"germany_forecasts_DishTS_CATBOOST_{horizon}m.csv")

if __name__ == "__main__":
  main()


Generating 12-month forecast...
Forecast saved to germany_forecasts_DishTS_CATBOOST_12m.csv

Generating 24-month forecast...
Forecast saved to germany_forecasts_DishTS_CATBOOST_24m.csv

Created files:
germany_forecasts_DishTS_CATBOOST_12m.csv
germany_forecasts_DishTS_CATBOOST_24m.csv


**Forecasts**
 - **Country: Japan**
 - **Forecast Horizons:12M and 24M Forecast**

In [9]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import XGBModel, CatBoostModel
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error
import torch.nn as nn
import torch.nn.functional as F

class DishTS(nn.Module):
  """DishTS Module for time series normalization"""
  def __init__(self, args):
      super().__init__()
      init = args['init']
      activate = True
      n_series = args['n_series']
      lookback = args['seq_len']

      if init == 'standard':
          self.reduce_mlayer = nn.Parameter(torch.rand(n_series, lookback, 2) / lookback)
      elif init == 'avg':
          self.reduce_mlayer = nn.Parameter(torch.ones(n_series, lookback, 2) / lookback)
      elif init == 'uniform':
          self.reduce_mlayer = nn.Parameter(torch.ones(n_series, lookback, 2) / lookback +
                                          torch.rand(n_series, lookback, 2) / lookback)

      self.gamma = nn.Parameter(torch.ones(n_series))
      self.beta = nn.Parameter(torch.zeros(n_series))
      self.activate = activate

  def forward(self, batch_x, mode='forward', dec_inp=None):
      if mode == 'forward':
          self.preget(batch_x)
          batch_x = self.forward_process(batch_x)
          dec_inp = None if dec_inp is None else self.forward_process(dec_inp)
          return batch_x, dec_inp
      elif mode == 'inverse':
          batch_y = self.inverse_process(batch_x)
          return batch_y

  def preget(self, batch_x):
      x_transpose = batch_x.permute(2, 0, 1)
      theta = torch.bmm(x_transpose, self.reduce_mlayer).permute(1, 2, 0)
      if self.activate:
          theta = F.gelu(theta)
      self.phil, self.phih = theta[:, :1, :], theta[:, 1:, :]
      self.xil = torch.sum(torch.pow(batch_x - self.phil, 2), axis=1, keepdim=True) / (batch_x.shape[1] - 1)
      self.xih = torch.sum(torch.pow(batch_x - self.phih, 2), axis=1, keepdim=True) / (batch_x.shape[1] - 1)

  def forward_process(self, batch_input):
      temp = (batch_input - self.phil) / torch.sqrt(self.xil + 1e-8)
      rst = temp.mul(self.gamma) + self.beta
      return rst

  def inverse_process(self, batch_input):
      return ((batch_input - self.beta) / self.gamma) * torch.sqrt(self.xih + 1e-8) + self.phih

class JAPANForecastGenerator:
  """Main class for generating forecasts using DishTS + XGB"""
  def __init__(self, data_path: str, random_seed: int = 1):
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      self.data = pd.read_csv(data_path)

      self.target_vars = [
          "Unemploymentrate",
          "RealbroadEER",
          "ShorttermIR",
          "OilpriceGlobalWTI",
          "CPIinflationrate"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int):
      """Prepare training data with correct lengths for past covariates"""
      # Split data
      train_size = len(self.data) - forecast_horizon
      train = self.data.head(train_size).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries using full training data
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(self.data[var]))

      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def generate_forecasts(self, forecast_horizons: list):
      """Generate forecasts for specified horizons"""
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data with correct lengths
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Initialize DishTS
          dish_args = {
              'init': 'standard',
              'n_series': len(self.target_vars),
              'seq_len': len(train_target_ts)
          }
          dish_ts = DishTS(dish_args)

          # Normalize training data
          train_normalized, _ = dish_ts(
              torch.tensor(train_target_ts.values(), dtype=torch.float32).unsqueeze(0)
          )

          # Create and train CatBoostModel with appropriate parameters
          model = CatBoostModel(
              lags = 6, #horizon,  # Set lags equal to horizon
              lags_past_covariates = 6, #horizon,  # Set past covariates lags equal to horizon
              output_chunk_length=horizon  # Set output chunk length equal to horizon
          )

          # Fit model with prepared data
          model.fit(
              TimeSeries.from_values(train_normalized.squeeze().detach().numpy()),
              past_covariates=train_exog_ts,
              # show_warnings=False
          )

          # Generate predictions
          pred_normalized = model.predict(horizon)

          # Inverse transform predictions
          with torch.no_grad():
              pred_original = dish_ts.inverse_process(
                  torch.tensor(pred_normalized.values(), dtype=torch.float32).unsqueeze(0)
              )

          # Create forecast DataFrame
          forecast_df = pd.DataFrame(
              np.squeeze(pred_original.detach().numpy()),
              columns=[f'forecast_{var.lower()}' for var in self.target_vars]
          )

          # Save forecasts
          output_file = f'japan_forecasts_DishTS_CATBOOST_{horizon}m.csv'
          forecast_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

          forecasts[horizon] = forecast_df

      return forecasts

def main():
  """Main function to run the forecast generation"""
  # Initialize forecast generator
  generator = JAPANForecastGenerator(
      data_path='all_mulvar_data_japan_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"japan_forecasts_DishTS_CATBOOST_{horizon}m.csv")

if __name__ == "__main__":
  main()


Generating 12-month forecast...
Forecast saved to japan_forecasts_DishTS_CATBOOST_12m.csv

Generating 24-month forecast...
Forecast saved to japan_forecasts_DishTS_CATBOOST_24m.csv

Created files:
japan_forecasts_DishTS_CATBOOST_12m.csv
japan_forecasts_DishTS_CATBOOST_24m.csv


**Forecasts**
 - **Country: UK**
 - **Forecast Horizons:12M and 24M Forecast**

In [10]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import XGBModel, CatBoostModel
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error
import torch.nn as nn
import torch.nn.functional as F

class DishTS(nn.Module):
  """DishTS Module for time series normalization"""
  def __init__(self, args):
      super().__init__()
      init = args['init']
      activate = True
      n_series = args['n_series']
      lookback = args['seq_len']

      if init == 'standard':
          self.reduce_mlayer = nn.Parameter(torch.rand(n_series, lookback, 2) / lookback)
      elif init == 'avg':
          self.reduce_mlayer = nn.Parameter(torch.ones(n_series, lookback, 2) / lookback)
      elif init == 'uniform':
          self.reduce_mlayer = nn.Parameter(torch.ones(n_series, lookback, 2) / lookback +
                                          torch.rand(n_series, lookback, 2) / lookback)

      self.gamma = nn.Parameter(torch.ones(n_series))
      self.beta = nn.Parameter(torch.zeros(n_series))
      self.activate = activate

  def forward(self, batch_x, mode='forward', dec_inp=None):
      if mode == 'forward':
          self.preget(batch_x)
          batch_x = self.forward_process(batch_x)
          dec_inp = None if dec_inp is None else self.forward_process(dec_inp)
          return batch_x, dec_inp
      elif mode == 'inverse':
          batch_y = self.inverse_process(batch_x)
          return batch_y

  def preget(self, batch_x):
      x_transpose = batch_x.permute(2, 0, 1)
      theta = torch.bmm(x_transpose, self.reduce_mlayer).permute(1, 2, 0)
      if self.activate:
          theta = F.gelu(theta)
      self.phil, self.phih = theta[:, :1, :], theta[:, 1:, :]
      self.xil = torch.sum(torch.pow(batch_x - self.phil, 2), axis=1, keepdim=True) / (batch_x.shape[1] - 1)
      self.xih = torch.sum(torch.pow(batch_x - self.phih, 2), axis=1, keepdim=True) / (batch_x.shape[1] - 1)

  def forward_process(self, batch_input):
      temp = (batch_input - self.phil) / torch.sqrt(self.xil + 1e-8)
      rst = temp.mul(self.gamma) + self.beta
      return rst

  def inverse_process(self, batch_input):
      return ((batch_input - self.beta) / self.gamma) * torch.sqrt(self.xih + 1e-8) + self.phih

class UKForecastGenerator:
  """Main class for generating forecasts using DishTS + XGB"""
  def __init__(self, data_path: str, random_seed: int = 1):
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      self.data = pd.read_csv(data_path)

      self.target_vars = [
          "Unemploymentrate",
          "RealbroadEER",
          "ShorttermIR",
          "OilpriceGlobalWTI",
          "CPIinflationrate"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int):
      """Prepare training data with correct lengths for past covariates"""
      # Split data
      train_size = len(self.data) - forecast_horizon
      train = self.data.head(train_size).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries using full training data
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(self.data[var]))

      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def generate_forecasts(self, forecast_horizons: list):
      """Generate forecasts for specified horizons"""
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data with correct lengths
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Initialize DishTS
          dish_args = {
              'init': 'standard',
              'n_series': len(self.target_vars),
              'seq_len': len(train_target_ts)
          }
          dish_ts = DishTS(dish_args)

          # Normalize training data
          train_normalized, _ = dish_ts(
              torch.tensor(train_target_ts.values(), dtype=torch.float32).unsqueeze(0)
          )

          # Create and train CatBoostModel with appropriate parameters
          model = CatBoostModel(
              lags = 6, #horizon,  # Set lags equal to horizon
              lags_past_covariates = 6, #horizon,  # Set past covariates lags equal to horizon
              output_chunk_length=horizon  # Set output chunk length equal to horizon
          )

          # Fit model with prepared data
          model.fit(
              TimeSeries.from_values(train_normalized.squeeze().detach().numpy()),
              past_covariates=train_exog_ts,
              # show_warnings=False
          )

          # Generate predictions
          pred_normalized = model.predict(horizon)

          # Inverse transform predictions
          with torch.no_grad():
              pred_original = dish_ts.inverse_process(
                  torch.tensor(pred_normalized.values(), dtype=torch.float32).unsqueeze(0)
              )

          # Create forecast DataFrame
          forecast_df = pd.DataFrame(
              np.squeeze(pred_original.detach().numpy()),
              columns=[f'forecast_{var.lower()}' for var in self.target_vars]
          )

          # Save forecasts
          output_file = f'uk_forecasts_DishTS_CATBOOST_{horizon}m.csv'
          forecast_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

          forecasts[horizon] = forecast_df

      return forecasts

def main():
  """Main function to run the forecast generation"""
  # Initialize forecast generator
  generator = UKForecastGenerator(
      data_path='all_mulvar_data_uk_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"uk_forecasts_DishTS_CATBOOST_{horizon}m.csv")

if __name__ == "__main__":
  main()


Generating 12-month forecast...
Forecast saved to uk_forecasts_DishTS_CATBOOST_12m.csv

Generating 24-month forecast...
Forecast saved to uk_forecasts_DishTS_CATBOOST_24m.csv

Created files:
uk_forecasts_DishTS_CATBOOST_12m.csv
uk_forecasts_DishTS_CATBOOST_24m.csv


**Forecasts**
 - **Country: Italy**
 - **Forecast Horizons:12M and 24M Forecast**

In [11]:
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import XGBModel, CatBoostModel
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error
import torch.nn as nn
import torch.nn.functional as F

class DishTS(nn.Module):
  """DishTS Module for time series normalization"""
  def __init__(self, args):
      super().__init__()
      init = args['init']
      activate = True
      n_series = args['n_series']
      lookback = args['seq_len']

      if init == 'standard':
          self.reduce_mlayer = nn.Parameter(torch.rand(n_series, lookback, 2) / lookback)
      elif init == 'avg':
          self.reduce_mlayer = nn.Parameter(torch.ones(n_series, lookback, 2) / lookback)
      elif init == 'uniform':
          self.reduce_mlayer = nn.Parameter(torch.ones(n_series, lookback, 2) / lookback +
                                          torch.rand(n_series, lookback, 2) / lookback)

      self.gamma = nn.Parameter(torch.ones(n_series))
      self.beta = nn.Parameter(torch.zeros(n_series))
      self.activate = activate

  def forward(self, batch_x, mode='forward', dec_inp=None):
      if mode == 'forward':
          self.preget(batch_x)
          batch_x = self.forward_process(batch_x)
          dec_inp = None if dec_inp is None else self.forward_process(dec_inp)
          return batch_x, dec_inp
      elif mode == 'inverse':
          batch_y = self.inverse_process(batch_x)
          return batch_y

  def preget(self, batch_x):
      x_transpose = batch_x.permute(2, 0, 1)
      theta = torch.bmm(x_transpose, self.reduce_mlayer).permute(1, 2, 0)
      if self.activate:
          theta = F.gelu(theta)
      self.phil, self.phih = theta[:, :1, :], theta[:, 1:, :]
      self.xil = torch.sum(torch.pow(batch_x - self.phil, 2), axis=1, keepdim=True) / (batch_x.shape[1] - 1)
      self.xih = torch.sum(torch.pow(batch_x - self.phih, 2), axis=1, keepdim=True) / (batch_x.shape[1] - 1)

  def forward_process(self, batch_input):
      temp = (batch_input - self.phil) / torch.sqrt(self.xil + 1e-8)
      rst = temp.mul(self.gamma) + self.beta
      return rst

  def inverse_process(self, batch_input):
      return ((batch_input - self.beta) / self.gamma) * torch.sqrt(self.xih + 1e-8) + self.phih

class ITALYForecastGenerator:
  """Main class for generating forecasts using DishTS + XGB"""
  def __init__(self, data_path: str, random_seed: int = 1):
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      self.data = pd.read_csv(data_path)

      self.target_vars = [
          "Unemploymentrate",
          "RealbroadEER",
          "ShorttermIR",
          "OilpriceGlobalWTI",
          "CPIinflationrate"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int):
      """Prepare training data with correct lengths for past covariates"""
      # Split data
      train_size = len(self.data) - forecast_horizon
      train = self.data.head(train_size).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries using full training data
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(self.data[var]))

      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def generate_forecasts(self, forecast_horizons: list):
      """Generate forecasts for specified horizons"""
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data with correct lengths
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Initialize DishTS
          dish_args = {
              'init': 'standard',
              'n_series': len(self.target_vars),
              'seq_len': len(train_target_ts)
          }
          dish_ts = DishTS(dish_args)

          # Normalize training data
          train_normalized, _ = dish_ts(
              torch.tensor(train_target_ts.values(), dtype=torch.float32).unsqueeze(0)
          )

          # Create and train CatBoostModel with appropriate parameters
          model = CatBoostModel(
              lags = 6, #horizon,  # Set lags equal to horizon
              lags_past_covariates = 6, #horizon,  # Set past covariates lags equal to horizon
              output_chunk_length=horizon  # Set output chunk length equal to horizon
          )

          # Fit model with prepared data
          model.fit(
              TimeSeries.from_values(train_normalized.squeeze().detach().numpy()),
              past_covariates=train_exog_ts,
              # show_warnings=False
          )

          # Generate predictions
          pred_normalized = model.predict(horizon)

          # Inverse transform predictions
          with torch.no_grad():
              pred_original = dish_ts.inverse_process(
                  torch.tensor(pred_normalized.values(), dtype=torch.float32).unsqueeze(0)
              )

          # Create forecast DataFrame
          forecast_df = pd.DataFrame(
              np.squeeze(pred_original.detach().numpy()),
              columns=[f'forecast_{var.lower()}' for var in self.target_vars]
          )

          # Save forecasts
          output_file = f'italy_forecasts_DishTS_CATBOOST_{horizon}m.csv'
          forecast_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

          forecasts[horizon] = forecast_df

      return forecasts

def main():
  """Main function to run the forecast generation"""
  # Initialize forecast generator
  generator = ITALYForecastGenerator(
      data_path='all_mulvar_data_italy_v2.csv'
  )

  # Generate forecasts for 12 and 24 months
  forecasts = generator.generate_forecasts([12, 24])

  # Print created files
  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"italy_forecasts_DishTS_CATBOOST_{horizon}m.csv")

if __name__ == "__main__":
  main()


Generating 12-month forecast...
Forecast saved to italy_forecasts_DishTS_CATBOOST_12m.csv

Generating 24-month forecast...
Forecast saved to italy_forecasts_DishTS_CATBOOST_24m.csv

Created files:
italy_forecasts_DishTS_CATBOOST_12m.csv
italy_forecasts_DishTS_CATBOOST_24m.csv
